# Logistic Regression Baseline

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

In [2]:
DATA_DIR = Path("data/merged")

train_df = pd.read_csv(DATA_DIR / "FPS_train.csv")
val_df = pd.read_csv(DATA_DIR / "FPS_val.csv")
test_df = pd.read_csv(DATA_DIR / "FPS_test.csv")

In [3]:
# define a logistic regression model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        C=1.2,
        max_iter=1000,
    ))
])

# Train without automatically rendering the fitted pipeline in Jupyter.
_ = model.fit(
    train_df["review"],
    train_df["voted_up"]
)

In [4]:
# evaluate model performance
def evaluate_model(model, data):
    y_true = data["voted_up"]
    y_pred = model.predict(data["review"])

    print(classification_report(
        y_true, y_pred, labels=[1,0], target_names=["recommended", "not recommend"],
        zero_division=0, digits=3
    ))

    print("Macro-F1:", f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0,
    ))
    
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[1,0]))

In [5]:
evaluate_model(model, val_df)

               precision    recall  f1-score   support

  recommended      0.871     0.756     0.809       884
not recommend      0.827     0.913     0.868      1133

     accuracy                          0.844      2017
    macro avg      0.849     0.834     0.839      2017
 weighted avg      0.846     0.844     0.842      2017

Macro-F1: 0.8385101108625019
Confusion matrix:
[[ 668  216]
 [  99 1034]]


In [6]:
print("Test set results:")
evaluate_model(model, test_df)

Test set results:
               precision    recall  f1-score   support

  recommended      0.872     0.779     0.823       882
not recommend      0.841     0.911     0.875      1136

     accuracy                          0.853      2018
    macro avg      0.857     0.845     0.849      2018
 weighted avg      0.855     0.853     0.852      2018

Macro-F1: 0.8488244137253811
Confusion matrix:
[[ 687  195]
 [ 101 1035]]


## External game tests

Run only the cells below in this notebook's trained kernel. They evaluate WARDOGS,
Cyberpunk 2077, and Persona 5 Royal using the current model and its training-time
text processing. These games are held out from training and parameter selection.


In [9]:
from config import EXTERNAL_TEST_GAME_IDS

external_test_dfs = {}
for game, appid in EXTERNAL_TEST_GAME_IDS.items():
    external_df = pd.read_csv(
        Path("data/processed") / f"reviews_{game}_{appid}.csv",
        usecols=["review", "voted_up"],
        dtype={"review": "string", "voted_up": "boolean"},
    )
    external_df = external_df.dropna(subset=["review", "voted_up"]).copy()
    external_df["voted_up"] = external_df["voted_up"].astype("int64")
    external_test_dfs[game] = external_df
    print(f"{game}: {len(external_df)} external test reviews")


WARDOGS: 1826 external test reviews
CYBERPUNK_2077: 1558 external test reviews
PERSONA_5_ROYAL: 1527 external test reviews


In [10]:
from sklearn.metrics import accuracy_score

# Reuse the fitted pipeline: predict transforms text without fitting TF-IDF again.
external_results = []
for game, external_df in external_test_dfs.items():
    y_true = external_df["voted_up"]
    y_pred = model.predict(external_df["review"])
    print(f"\n{game}")
    print(classification_report(
        y_true, y_pred, labels=[1, 0],
        target_names=["recommended", "not recommended"],
        zero_division=0, digits=3,
    ))
    print("Confusion matrix (rows=true, columns=predicted; order=[1, 0]):")
    print(confusion_matrix(y_true, y_pred, labels=[1, 0]))
    external_results.append({
        "game": game,
        "samples": len(external_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    })

external_results = pd.DataFrame(external_results).set_index("game")
print("\nExternal test summary:")
print(external_results.to_string(float_format=lambda value: f"{value:.4f}"))



WARDOGS
                 precision    recall  f1-score   support

    recommended      0.935     0.600     0.731       883
not recommended      0.720     0.961     0.823       943

       accuracy                          0.786      1826
      macro avg      0.827     0.780     0.777      1826
   weighted avg      0.824     0.786     0.778      1826

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[530 353]
 [ 37 906]]

CYBERPUNK_2077
                 precision    recall  f1-score   support

    recommended      0.849     0.700     0.767       666
not recommended      0.802     0.907     0.851       892

       accuracy                          0.818      1558
      macro avg      0.825     0.803     0.809      1558
   weighted avg      0.822     0.818     0.815      1558

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[466 200]
 [ 83 809]]

PERSONA_5_ROYAL
                 precision    recall  f1-score   support

    recommended      0.794     0.679  